# Hands-On verl Fully Async Policy DAPO Training on AMD Instinct™ GPUs

In [A Deep Dive into verl Fully Async Training](https://github.com/Vivicai1005/verl-rocm-tutorials/blob/main/verl-fully-async-policy-training-zh.md), we walked through the verl Fully Async Policy architecture from the source code and explained how Rollout, Training, and Parameter Synchronization work together.

Now that we understand the overall architecture, we can move on to practice and run a complete Fully Async Policy DAPO training job on AMD Instinct™ GPUs.

verl provides an example script, [dapo_7b_math_fsdp2_4_4.sh](https://github.com/verl-project/verl/blob/main/verl/experimental/fully_async_policy/shell/dapo_7b_math_fsdp2_4_4.sh). It runs on a single 8-GPU node and divides the GPUs into two independent groups:
- 4 GPUs for Training: responsible for Actor forward/backward passes, optimizer updates, and other training computations;
- 4 GPUs for Rollout: running the vLLM inference engine to continuously generate rollout samples.

Because Training and Rollout use separate GPU resources, rollout generation does not need to wait for every policy update to finish. The two compute pipelines can keep running in parallel, which is one of the core characteristics of Fully Async Training.

In this tutorial, we will run the same DAPO training architecture on AMD Instinct™ GPUs. To reduce the hardware requirement, we take advantage of the large memory capacity of AMD Instinct™ GPUs and scale the official  **4 Training GPUs + 4 Rollout GPUs** configuration down to **2 Training GPUs + 2 Rollout GPUs**. This allows us to complete the full Fully Async DAPO training workflow with only 4 GPUs.

Starting from `dapo_7b_math_fsdp2_4_4.sh`, we will work through the following steps:
1. Understand the DAPO algorithm and its key improvements over GRPO;
2. Verify the AMD ROCm and verl runtime environment;
3. Configure Fully Async DAPO Training with 2 GPUs for Training and 2 GPUs for Rollout, and understand the key parameters involved;
4. Analyze training logs and metrics to verify that Fully Async Training is running as expected.


## What is DAPO

[DAPO (Decoupled Clip and Dynamic sAmpling Policy Optimization)](https://arxiv.org/pdf/2503.14476) builds on GRPO (Group Relative Policy Optimization) and introduces a series of improvements for issues commonly seen in large-scale Long-CoT RL training, including training instability, entropy collapse, ineffective samples, and overly long responses.

### GRPO: Computing Relative Advantage from Multiple Responses to the Same Prompt
For a given prompt, GRPO does not generate only one response. Instead, it samples a group of responses from the current policy and computes a reward for each response:

$$
R_i = r(q, o_i), \qquad i = 1,2,\dots,G
$$

Here, \(q\) denotes the prompt, \(o_i\) is the \(i\)-th response, and \(R_i\) is the reward assigned to that response.

Unlike standard PPO, which typically requires a separately trained Critic / Value Model, GRPO can estimate advantage directly from the relative rewards of responses generated for the same prompt, without training an additional Value Model:

$$
\hat{A}_i =
\frac{
R_i - \operatorname{mean}(R_1,\dots,R_G)
}{
\operatorname{std}(R_1,\dots,R_G)
}
$$

A response receives a positive advantage if its reward is above the group average, and a negative advantage if its reward is below the group average. Once the advantage is computed, GRPO uses it to update the policy. For the \(t\)-th token in response \(o_i\), it first computes the probability ratio between the new and old policies:

$$
r_{i,t}(\theta)
=
\frac{
\pi_\theta(o_{i,t} \mid q, o_{i,<t})
}{
\pi_{\theta_{\text{old}}}(o_{i,t} \mid q, o_{i,<t})
}
$$

The advantage then weights the direction of the policy update. In simplified form:

$$
L_{i,t} \propto r_{i,t}(\theta)\hat{A}_i
$$

All tokens in the same response share that response's group-relative advantage \($\hat{A}_i\$).

Therefore:

- If $\hat{A}_i > 0$, the response has a reward above the group average, so training increases the probability of generating its tokens;
- If $\hat{A}_i < 0$, the response has a reward below the group average, so training decreases the probability of generating those tokens;
- If $\hat{A}_i \approx 0$, the response contributes little useful training signal to the current policy update.

In practice, GRPO also uses clipping, similar to PPO, to constrain the size of each policy update:

$$
L_{\text{GRPO}}
=
\frac{1}{G}
\sum_{i=1}^{G}
\frac{1}{|o_i|}
\sum_{t=1}^{|o_i|}
\min
\left(
r_{i,t}(\theta)\hat{A}_i,\,
\operatorname{clip}
\left(
r_{i,t}(\theta),
1-\epsilon,
1+\epsilon
\right)
\hat{A}_i
\right)
$$

### DAPO: Improvements for Long-CoT Training on Top of GRPO
DAPO keeps the core idea of GRPO: sample multiple responses for the same prompt, compute relative advantages from within-group rewards, and use those advantages to update the policy.

On top of GRPO, DAPO introduces several improvements specifically for Long-CoT RL training. In this tutorial, we focus on the designs used by our current configuration: Clip-Higher, Token-Level Policy Gradient Loss, and Overlong Reward Shaping.

#### 1. Clip-Higher: Encouraging More Exploration
GRPO typically uses symmetric clipping:

$$
[1-\epsilon,\ 1+\epsilon]
$$

For example, when $\epsilon=0.2$:

$$
[0.8,\ 1.2]
$$

This prevents a single policy update from becoming too large, but it also limits how much low-probability tokens can increase. Over time, the policy may become increasingly concentrated on existing reasoning paths, potentially leading to entropy collapse.

DAPO decouples the lower and upper clipping bounds:

$$
[1-\epsilon_{\text{low}},\ 1+\epsilon_{\text{high}}]
$$

The paper uses:

$$
\epsilon_{\text{low}}=0.2,\qquad
\epsilon_{\text{high}}=0.28
$$

This raises the upper bound from 1.2 to 1.28. As a result, low-probability tokens with positive advantage have more room to increase in probability, encouraging the model to explore new reasoning paths.

#### 2. Token-Level Policy Gradient Loss: Aggregating Loss by Token
GRPO typically averages token losses within each response first, and then averages across responses:

$$
\frac{1}{G}
\sum_{i=1}^{G}
\frac{1}{|o_i|}
\sum_{t=1}^{|o_i|}
L_{i,t}
$$

This means that each response receives roughly the same weight in the final loss regardless of its length. In Long-CoT training, this can dilute the training signal of individual tokens in longer responses.

DAPO instead uses a token-level mean:

$$
\frac{
\sum_i \sum_t L_{i,t}
}{
\sum_i |o_i|
}
$$

In other words, it averages directly over all valid response tokens in the batch. This gives each token a more consistent contribution to the policy update and is better suited to Long-CoT training.

#### 3. Overlong Reward Shaping: Reducing Reward Noise from Overly Long Responses
Long-CoT responses may be truncated when they reach `max_response_length`. In such cases, the reasoning process itself may still be valid, but the model may simply not have had enough room to produce the final answer. Treating every truncated response as fully incorrect and assigning it a strong negative reward can introduce misleading training signals.

DAPO therefore uses Soft Overlong Punishment. Instead of applying a sudden penalty only after a response reaches the maximum length, it gradually increases the length penalty as the response approaches `max_response_length`. Responses within the normal length range receive no extra penalty; once the response enters the configured overlong buffer, the penalty increases smoothly with response length.

This design introduces a gradual length penalty before the response reaches the hard maximum, encouraging the model to learn to control reasoning length. Compared with a sudden reward change only after truncation occurs, this reduces truncation-related reward noise and improves the stability of Long-CoT RL training.


## Verify the AMD ROCm and verl Runtime Environment
Before starting training, first make sure the current container can detect the AMD GPUs correctly and verify that the ROCm, verl, and vLLM environments are working as expected.


### Check the AMD GPUs
First, use `amd-smi` to check the AMD GPUs visible inside the container:


In [ ]:
!amd-smi

This tutorial uses 4 AMD Instinct™ GPU devices. Under normal conditions, `amd-smi` or `rocm-smi` should detect all 4 GPUs and display information such as memory usage, temperature, and device status.


### Load the verl Python Environment and Verify Dependencies
The verl container comes with a preconfigured `/opt/venv` Python environment. Activate it in the current shell and verify that verl and vLLM can be imported successfully.


In [ ]:
%%bash
set -e

source /opt/venv/bin/activate

# Map the vLLM source directory
ln -sfn /workspace/verl-test/vllm /workspace/vllm

echo "Python:"
which python
python --version

python - <<'PY'
import verl
import vllm

print("verl version :", verl.__version__)
print("verl module :", verl.__file__)
print("vLLM version:", vllm.__version__)
print("vLLM module :", vllm.__file__)
PY

## Prepare and Configure Fully Async DAPO Training
Next, we will download the model used for this run, prepare the DAPO dataset, and configure Fully Async DAPO Training with 2 GPUs for Training and 2 GPUs for Rollout. Along the way, we will walk through the most important training parameters in the launch configuration.


### Download the Model
This tutorial uses [Qwen2.5-Math-7B](https://huggingface.co/Qwen/Qwen2.5-Math-7B) as the base model. Run the following command to download it to `/root/verl/models/Qwen2.5-Math-7B`:


In [ ]:
%%bash
set -e

hf download Qwen/Qwen2.5-Math-7B --local-dir /root/verl/models/Qwen2.5-Math-7B

The default maximum position embedding length in Qwen2.5-Math-7B is not large enough for the long sequences used in this training run. After downloading the model, update `max_position_embeddings` in `config.json` to `32768`:


In [ ]:
%%bash
set -e

MODEL_PATH=/root/verl/models/Qwen2.5-Math-7B

python - <<'PY'
import json
from pathlib import Path

config_path = Path("/root/verl/models/Qwen2.5-Math-7B/config.json")

with config_path.open("r", encoding="utf-8") as f:
    config = json.load(f)

old_value = config.get("max_position_embeddings")
config["max_position_embeddings"] = 32768

with config_path.open("w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
    f.write("\n")

print(f"Updated max_position_embeddings: {old_value} -> 32768")
PY

### Prepare the Training and Validation Data
Download and run the DAPO data preparation script provided by verl-recipe. The script prepares the DAPO-Math-17k training set and the AIME 2024 validation set used in this tutorial:


In [ ]:
%%bash
set -e

source /opt/venv/bin/activate

wget -q -O prepare_dapo_data.sh \
    https://raw.githubusercontent.com/verl-project/verl-recipe/main/dapo/prepare_dapo_data.sh

# Force wget inside the script to use a single-line progress bar
wget() {
    command wget --progress=bar:force:noscroll "$@"
}
export -f wget

bash prepare_dapo_data.sh

After the data preparation finishes, the following two files will be available:

- `/root/verl/data/dapo-math-17k.parquet`: training dataset;
- `/root/verl/data/aime-2024.parquet`: validation dataset.


### Understand the Key Fully Async DAPO Training Parameters
After preparing the model and datasets, let's walk through the key parameters used to launch Fully Async DAPO Training.

#### Configure Model and Data Paths

First, specify the model, training dataset, validation dataset, and checkpoint directory:

```bash
RAY_DATA_HOME=${RAY_DATA_HOME:-"${HOME}/verl"}

MODEL_PATH=${MODEL_PATH:-"${RAY_DATA_HOME}/models/Qwen2.5-Math-7B"}
CKPTS_DIR=${CKPTS_DIR:-"${RAY_DATA_HOME}/ckpts/${project_name}/${exp_name}"}

TRAIN_FILE=${TRAIN_FILE:-"${RAY_DATA_HOME}/data/dapo-math-17k.parquet"}
TEST_FILE=${TEST_FILE:-"${RAY_DATA_HOME}/data/aime-2024.parquet"}
```

This run uses **Qwen2.5-Math-7B** as the base model, **DAPO-Math-17k** as the training set, and **AIME 2024** to evaluate model performance during training.

#### Split GPUs Between Training and Rollout
This run uses 4 GPUs on a single node, assigning 2 to Rollout and the remaining 2 to Training:

```bash
NNODES=${NNODES:-1}
NGPUS_PER_NODE=${NGPUS_PER_NODE:-4}

n_gpus_rollout=${N_GPUS_ROLLOUT:-2}
n_gpus_training=$((NGPUS_PER_NODE - n_gpus_rollout))
```

#### Configure the Rollout Engine
The Rollout side uses vLLM in asynchronous inference mode:

```bash
rollout_mode="async"
rollout_name="vllm"

export VLLM_USE_V1=1
return_raw_chat="True"
```

#### Prompt and Response Lengths
The maximum Prompt and Response lengths are configured as follows:
```bash
max_prompt_length=$((1024 * 2))
max_response_length=$((1024 * 8))
```

#### Configure DAPO Algorithm Parameters
##### Group Relative Policy Optimization
```bash
adv_estimator=grpo
n_resp_per_prompt=16
```
`adv_estimator=grpo` means that GRPO is used to estimate advantages. The Rollout Engine generates 16 Responses for each Prompt and computes relative advantages from the rewards within the same response group.

##### Asymmetric Clipping
```bash
clip_ratio_low=0.2
clip_ratio_high=0.28
```
DAPO configures the lower and upper clipping bounds separately. The wider upper bound gives tokens with positive advantage more room to increase in probability, which helps mitigate entropy collapse and encourages policy exploration during training.

##### Overlong Reward Shaping
```bash
enable_overlong_buffer=True
overlong_buffer_len=$((1024 * 4))
overlong_penalty_factor=1.0
```
Overlong Reward Shaping penalizes overly long Responses. Instead of applying a fixed penalty only after the length limit is exceeded, it gradually increases the penalty within the configured buffer range, producing a smoother reward transition near the length boundary.

##### Token-Level Loss Aggregation
```bash
loss_agg_mode="token-mean"
```
`token-mean` aggregates the loss over all valid Response tokens in a mini-batch and then divides by the total number of valid tokens, giving every valid token equal weight in the final loss.

#### Fully Async Training Parameters
```bash
train_prompt_bsz=0
gen_prompt_bsz=1
train_prompt_mini_bsz=32

staleness_threshold=0.1
trigger_parameter_sync_step=4
require_batches=4
partial_rollout=True
```
Together, these parameters control the cadence of sample generation by the Rollouter, sample consumption by the Trainer, Actor updates, and parameter synchronization.

Fully Async Training does not use a conventional fixed global batch, so `train_prompt_bsz` is set to 0. `gen_prompt_bsz=1` means the Rollouter reads and processes one Prompt at a time, enabling streaming-style training through sample-by-sample production and consumption.

`require_batches=4` means that the Trainer collects `4 × 32 = 128` Prompts before each local update. Because each Prompt generates 16 Responses, one update contains 2048 Response trajectories.

With `trigger_parameter_sync_step=4`, the Trainer synchronizes the latest Actor parameters to the Rollout Engine after every 4 local updates, or after processing 512 Prompts.

Because the Rollouter and Trainer run concurrently in Fully Async Training, the Trainer may receive samples generated by an older Policy version. `staleness_threshold=0.1` allows the system to use a controlled proportion of stale samples so the Rollouter can generate data slightly ahead of the Trainer and reduce waiting time. A smaller value keeps training closer to on-policy, while a larger value increases asynchrony but also increases the gap between the Training Policy and the Rollout Policy used to generate samples.

When parameter synchronization begins, the Rollout Engine may still have Responses that have not finished generating. With Partial Rollout enabled through `partial_rollout=True`, the system can pause those rollouts, preserve the partially generated results, and resume generation after synchronization. This reduces pipeline bubbles caused by waiting for long Responses to finish. `partial_rollout` only takes effect when `staleness_threshold > 0`.

#### Rollout Sampling Parameters
```bash
temperature=1.0
top_p=1.0
top_k=-1
val_top_p=0.7

actor_rollout_ref.rollout.val_kwargs.top_p=${val_top_p}
actor_rollout_ref.rollout.val_kwargs.do_sample=True
actor_rollout_ref.rollout.val_kwargs.n=1
```
During training, `temperature=1.0` and `top_p=1.0` preserve a high degree of sampling diversity, allowing the same Prompt to produce varied Responses and providing enough exploration for GRPO's within-group reward comparison.
During validation, `top_p=0.7` is used and only one Response is generated per Prompt.

#### Rollout Log Probability
```bash
actor_rollout_ref.rollout.calculate_log_probs=True
```
In Fully Async Training, the Rollout Policy may lag behind the Training Policy. The Rollout Engine therefore records the log probability used when each token is generated, ensuring that `old_log_prob` matches the Policy version that actually generated the sample.

#### FSDP2 and Parallelism Configuration
```bash
fsdp_size=2
gen_tp=1
sp_size=1

ref_offload=True
actor_offload=False
```
The Actor uses FSDP2, with `fsdp_size=2` sharding model parameters across the 2 Training GPUs.
The Rollout Engine uses a Tensor Parallel Size of 1, meaning each vLLM replica runs on one GPU. `sp_size=1` means Ulysses Sequence Parallelism is disabled.

#### Dynamic Batch and Token Budget
```bash
use_dynamic_bsz=True

actor_ppo_max_token_len=$(((max_prompt_length + max_response_length) * 2))
infer_ppo_max_token_len=$(((max_prompt_length + max_response_length) * 3))
```
With Dynamic Batch enabled, the system forms micro-batches based on the actual number of tokens in each sequence rather than only using a fixed number of sequences per batch, reducing unnecessary computation caused by padding.

#### vLLM Memory and Prefill Configuration
```bash
actor_rollout_ref.rollout.gpu_memory_utilization=0.80
actor_rollout_ref.rollout.enable_chunked_prefill=True
actor_rollout_ref.rollout.max_num_batched_tokens=6144
```
vLLM can use up to roughly 80% of GPU memory and uses Chunked Prefill to process longer Prompts in chunks. `max_num_batched_tokens=6144` limits the maximum number of tokens that can be processed in a single vLLM scheduling step, helping control peak memory usage on the Rollout side.

#### Actor Optimizer Parameters
```bash
actor_rollout_ref.actor.optim.lr=1e-6
actor_rollout_ref.actor.optim.lr_warmup_steps=10
actor_rollout_ref.actor.optim.weight_decay=0.1

actor_rollout_ref.actor.entropy_coeff=0
actor_rollout_ref.actor.grad_clip=1.0
```
The Actor uses a learning rate of `1e-6`, 10 warmup steps, and `0.1` weight decay. `entropy_coeff=0` means no additional entropy bonus is applied, while `grad_clip=1.0` limits the gradient norm to help prevent unstable updates caused by excessively large gradients.

#### Training Scale and Validation
```bash
total_rollout_steps=$((512 * 100))
test_freq=10

rollout.total_rollout_steps=51200
trainer.total_epochs=10
trainer.test_freq=10
trainer.val_before_train=True
```
This run generates up to `51200` Rollout Prompt samples and trains for up to 10 epochs over the dataset. A validation run on AIME 2024 is performed before training begins, followed by validation every 10 training steps.

#### Logging and Checkpoints
```bash
trainer.logger=['console','wandb']
trainer.default_local_dir="${CKPTS_DIR}"
trainer.resume_mode=auto
trainer.save_freq=-1
```
Training metrics are written to both the terminal and Weights & Biases (W&B), where you can monitor reward, loss, Response length, and asynchronous training status in real time. `resume_mode=auto` automatically checks the checkpoint directory at startup and attempts to resume training. `save_freq=-1` disables periodic checkpoint saving. To enable checkpointing, set `save_freq` to a positive integer. For example, `save_freq=50` saves a checkpoint to `CKPTS_DIR` every 50 training steps.


## Start Training and Monitor Training Metrics


### Configure a W&B API Key (Optional)
If you want to log training metrics to Weights & Biases, enter your W&B API Key securely below. The value you type will not be displayed in the Notebook. Press Enter without entering a key to use Console logging only.


In [ ]:
import getpass
import os

if os.environ.get("WANDB_API_KEY"):
    print("WANDB_API_KEY is already configured.")
else:
    wandb_api_key = getpass.getpass(
        "Enter WANDB_API_KEY (press Enter to skip): "
    ).strip()

    if wandb_api_key:
        os.environ["WANDB_API_KEY"] = wandb_api_key
        print("WANDB_API_KEY configured. W&B logging will be enabled.")
    else:
        print("WANDB_API_KEY not provided. Console logging will be used.")

### Start Training


In [ ]:
%%bash
set -euo pipefail

RUN_NAME="dapo_qwen2.5_7b_fully_async_2_2"
LOG_DIR="${HOME}/verl/logs"
LOG_FILE="${LOG_DIR}/${RUN_NAME}.log"
PID_FILE="${LOG_DIR}/${RUN_NAME}.pid"

mkdir -p "${LOG_DIR}"

# Prevent duplicate training jobs
if [[ -f "${PID_FILE}" ]]; then
    existing_pid=$(cat "${PID_FILE}")

    if kill -0 "${existing_pid}" 2>/dev/null; then
        echo "Training is already running."
        echo "PID: ${existing_pid}"
        echo "Log: ${LOG_FILE}"
        exit 1
    else
        echo "Removing stale PID file: ${PID_FILE}"
        rm -f "${PID_FILE}"
    fi
fi

nohup setsid bash -s >"${LOG_FILE}" 2>&1 <<'TRAIN_SCRIPT' &
#!/usr/bin/env bash
set -euo pipefail

# Activate Python virtual environment
source /opt/venv/bin/activate

# Write Python logs immediately
export PYTHONUNBUFFERED=1

# Select logger according to WANDB_API_KEY
if [[ -n "${WANDB_API_KEY:-}" ]]; then
    trainer_logger="['console','wandb']"
    echo "WANDB_API_KEY detected. W&B logging is enabled."
else
    trainer_logger="['console']"
    echo "WANDB_API_KEY not found. Console logging is enabled."
fi

project_name='DAPO'
exp_name='DAPO-Qwen2.5-7b-MATH-fsdp2-fully-async-2-2'

# Paths
RAY_DATA_HOME=${RAY_DATA_HOME:-"${HOME}/verl"}

MODEL_PATH=${MODEL_PATH:-"${RAY_DATA_HOME}/models/Qwen2.5-Math-7B"}
CKPTS_DIR=${CKPTS_DIR:-"${RAY_DATA_HOME}/ckpts/${project_name}/${exp_name}"}
TRAIN_FILE=${TRAIN_FILE:-"${RAY_DATA_HOME}/data/dapo-math-17k.parquet"}
TEST_FILE=${TEST_FILE:-"${RAY_DATA_HOME}/data/aime-2024.parquet"}

# Rollout engine
rollout_mode="async"
rollout_name="vllm"

if [[ "${rollout_mode}" == "async" ]]; then
    export VLLM_USE_V1=1
    return_raw_chat="True"
else
    return_raw_chat="False"
fi

# Algorithm parameters
adv_estimator="grpo"

use_kl_in_reward=False
kl_coef=0.0
use_kl_loss=False
kl_loss_coef=0.0

clip_ratio_low=0.2
clip_ratio_high=0.28

# Prompt and response length
max_prompt_length=$((1024 * 2))
max_response_length=$((1024 * 8))

enable_overlong_buffer=True
overlong_buffer_len=$((1024 * 4))
overlong_penalty_factor=1.0

# Loss aggregation
loss_agg_mode="token-mean"

# Sampling parameters
temperature=1.0
top_p=1.0
top_k=-1
val_top_p=0.7

# Performance parameters
use_dynamic_bsz=True

actor_ppo_max_token_len=$(((max_prompt_length + max_response_length) * 2))
infer_ppo_max_token_len=$(((max_prompt_length + max_response_length) * 3))

ref_offload=True
actor_offload=False

gen_tp=1
sp_size=1
fsdp_size=2

# GPU resource allocation
NNODES=${NNODES:-1}
NGPUS_PER_NODE=${NGPUS_PER_NODE:-4}

n_gpus_rollout=${N_GPUS_ROLLOUT:-2}
n_gpus_training=$((NGPUS_PER_NODE - n_gpus_rollout))

# Fully Async parameters
train_prompt_bsz=0
gen_prompt_bsz=1
n_resp_per_prompt=16
train_prompt_mini_bsz=32

total_rollout_steps=$((512 * 100))
test_freq=10

staleness_threshold=0.1
trigger_parameter_sync_step=4
require_batches=4
partial_rollout=True

echo "Starting Fully Async DAPO Training..."
echo "Project: ${project_name}"
echo "Experiment: ${exp_name}"
echo "Model: ${MODEL_PATH}"
echo "Training GPUs: ${n_gpus_training}"
echo "Rollout GPUs: ${n_gpus_rollout}"
echo "Logger: ${trainer_logger}"

python -m verl.experimental.fully_async_policy.fully_async_main \
    data.train_files="${TRAIN_FILE}" \
    data.val_files="${TEST_FILE}" \
    data.prompt_key=prompt \
    data.truncation='left' \
    data.max_prompt_length=${max_prompt_length} \
    data.max_response_length=${max_response_length} \
    data.train_batch_size=${train_prompt_bsz} \
    data.gen_batch_size=${gen_prompt_bsz} \
    data.return_raw_chat=${return_raw_chat} \
    actor_rollout_ref.rollout.n=${n_resp_per_prompt} \
    algorithm.adv_estimator=${adv_estimator} \
    algorithm.use_kl_in_reward=${use_kl_in_reward} \
    algorithm.kl_ctrl.kl_coef=${kl_coef} \
    actor_rollout_ref.actor.fsdp_config.strategy=fsdp2 \
    critic.strategy=fsdp2 \
    actor_rollout_ref.actor.use_kl_loss=${use_kl_loss} \
    actor_rollout_ref.actor.kl_loss_coef=${kl_loss_coef} \
    actor_rollout_ref.actor.clip_ratio_low=${clip_ratio_low} \
    actor_rollout_ref.actor.clip_ratio_high=${clip_ratio_high} \
    actor_rollout_ref.actor.clip_ratio_c=10.0 \
    actor_rollout_ref.model.use_remove_padding=True \
    actor_rollout_ref.hybrid_engine=False \
    +actor_rollout_ref.model.override_config.max_position_embeddings=32768 \
    actor_rollout_ref.actor.use_dynamic_bsz=${use_dynamic_bsz} \
    actor_rollout_ref.ref.log_prob_use_dynamic_bsz=${use_dynamic_bsz} \
    actor_rollout_ref.rollout.log_prob_use_dynamic_bsz=${use_dynamic_bsz} \
    actor_rollout_ref.actor.ppo_max_token_len_per_gpu=${actor_ppo_max_token_len} \
    actor_rollout_ref.ref.log_prob_max_token_len_per_gpu=${infer_ppo_max_token_len} \
    actor_rollout_ref.rollout.log_prob_max_token_len_per_gpu=${infer_ppo_max_token_len} \
    actor_rollout_ref.model.path="${MODEL_PATH}" \
    actor_rollout_ref.actor.optim.lr=1e-6 \
    actor_rollout_ref.actor.optim.lr_warmup_steps=10 \
    actor_rollout_ref.actor.optim.weight_decay=0.1 \
    actor_rollout_ref.actor.ppo_mini_batch_size=${train_prompt_mini_bsz} \
    actor_rollout_ref.actor.fsdp_config.param_offload=${actor_offload} \
    actor_rollout_ref.actor.fsdp_config.optimizer_offload=${actor_offload} \
    actor_rollout_ref.actor.entropy_coeff=0 \
    actor_rollout_ref.actor.grad_clip=1.0 \
    actor_rollout_ref.actor.loss_agg_mode=${loss_agg_mode} \
    actor_rollout_ref.actor.ulysses_sequence_parallel_size=${sp_size} \
    actor_rollout_ref.rollout.gpu_memory_utilization=0.80 \
    actor_rollout_ref.rollout.tensor_model_parallel_size=${gen_tp} \
    actor_rollout_ref.rollout.enable_chunked_prefill=True \
    actor_rollout_ref.rollout.max_num_batched_tokens=$((max_prompt_length + max_response_length)) \
    actor_rollout_ref.rollout.temperature=${temperature} \
    actor_rollout_ref.rollout.top_p=${top_p} \
    actor_rollout_ref.rollout.top_k=${top_k} \
    actor_rollout_ref.rollout.val_kwargs.temperature=${temperature} \
    actor_rollout_ref.rollout.val_kwargs.top_p=${val_top_p} \
    actor_rollout_ref.rollout.val_kwargs.top_k=${top_k} \
    actor_rollout_ref.rollout.val_kwargs.do_sample=True \
    actor_rollout_ref.rollout.val_kwargs.n=1 \
    actor_rollout_ref.rollout.calculate_log_probs=True \
    actor_rollout_ref.ref.fsdp_config.param_offload=${ref_offload} \
    actor_rollout_ref.ref.ulysses_sequence_parallel_size=${sp_size} \
    actor_rollout_ref.actor.fsdp_config.fsdp_size=${fsdp_size} \
    actor_rollout_ref.rollout.name=${rollout_name} \
    actor_rollout_ref.rollout.mode=${rollout_mode} \
    reward.reward_manager.name=dapo \
    +reward.reward_kwargs.overlong_buffer_cfg.enable=${enable_overlong_buffer} \
    +reward.reward_kwargs.overlong_buffer_cfg.len=${overlong_buffer_len} \
    +reward.reward_kwargs.overlong_buffer_cfg.penalty_factor=${overlong_penalty_factor} \
    +reward.reward_kwargs.overlong_buffer_cfg.log=False \
    +reward.reward_kwargs.max_resp_len=${max_response_length} \
    trainer.logger="${trainer_logger}" \
    trainer.project_name="${project_name}" \
    trainer.experiment_name="${exp_name}" \
    trainer.val_before_train=True \
    trainer.save_freq=-1 \
    trainer.default_local_dir="${CKPTS_DIR}" \
    trainer.resume_mode=auto \
    trainer.nnodes="${NNODES}" \
    trainer.n_gpus_per_node="${n_gpus_training}" \
    rollout.nnodes="${NNODES}" \
    rollout.n_gpus_per_node="${n_gpus_rollout}" \
    rollout.total_rollout_steps="${total_rollout_steps}" \
    trainer.total_epochs=10 \
    trainer.test_freq="${test_freq}" \
    async_training.staleness_threshold="${staleness_threshold}" \
    async_training.trigger_parameter_sync_step="${trigger_parameter_sync_step}" \
    async_training.require_batches="${require_batches}" \
    async_training.partial_rollout="${partial_rollout}"

TRAIN_SCRIPT

training_pid=$!
echo "${training_pid}" >"${PID_FILE}"
disown "${training_pid}" 2>/dev/null || true

echo "Training started in the background."
echo "PID: ${training_pid}"
echo "Log: ${LOG_FILE}"
echo "PID file: ${PID_FILE}"

### Follow the Training Log in Real Time
To continuously stream newly generated log output, run:


In [ ]:
%%bash

RUN_NAME="dapo_qwen2.5_7b_fully_async_2_2"
LOG_FILE="${HOME}/verl/logs/${RUN_NAME}.log"

if [[ ! -f "${LOG_FILE}" ]]; then
    echo "Log file does not exist: ${LOG_FILE}"
    exit 1
fi

tail -n 100 -f "${LOG_FILE}"

> **Tip**
>
> When you are done viewing the log, click the **Stop (■)** button in the Notebook toolbar to stop output from the current cell.
>
> This only stops the live log stream. It does **not** stop the training job running in the background. To view the log again, simply rerun the cell above.


### Understand the Key Metrics in the Training Log
Once training starts, verl reports a set of metrics at every Training Step. These metrics reflect the state of the Fully Async Pipeline, the stability of Policy Updates, the quality of Rollout Samples, and system resource utilization. Below, we group the most important metrics by category.

#### Training Progress and Parameter Version

| Metric | Example | Meaning |
| --- | ---: | --- |
| `training/global_step` | `399` | Cumulative number of Training Steps completed by the Actor |
| `training/epoch` | `0` | Current training epoch |
| `fully_async/count/current_param_version` | `99` | Policy Parameter Version currently used on the Rollout side |

This run uses:

```text
trigger_parameter_sync_step = 4
```

This means the Actor synchronizes the latest parameters to the Rollout Engines every 4 Training Steps. Therefore, when `global_step=399`, a `current_param_version` of about `99` is consistent with the expected synchronization cadence and indicates that parameter synchronization is progressing normally.

#### Fully Async Pipeline State

| Metric | Example | Meaning |
| --- | ---: | --- |
| `active_tasks_size` | `32` | Number of generation tasks currently being processed by the Rollouter |
| `max_concurrent_samples` | `32` | Maximum number of tasks the Rollouter can process concurrently |
| `pending_queue_size` | `128` | Number of tasks waiting to be processed by the Rollouter |
| `mq_queue_size` | `159.75` | Number of samples in the MessageQueue waiting to be consumed by the Trainer |
| `required_samples` | `128` | Number of samples required by the Trainer for each training update |
| `total_generated_samples` | `50943` | Cumulative number of samples generated by the Rollouter |
| `total_wait_time` | `51.23 s` | Cumulative time the Trainer has spent waiting for training samples |

This run is configured with:

```text
require_batches     = 4
ppo_mini_batch_size = 32
```

Therefore, each training update requires:

```text
required_samples
= require_batches × ppo_mini_batch_size
= 4 × 32
= 128
```

Here, `mq_queue_size` is greater than `required_samples`, which means the MessageQueue currently contains enough data for the Trainer to consume. `active_tasks_size` has also reached `max_concurrent_samples`, indicating that the Rollouter is fully utilizing the configured generation concurrency.

Metrics such as `mq_queue_size` may be aggregated or averaged across multiple Workers, so they can appear as fractional values in the log.

#### Rollout Processing Time

| Metric | Example | Meaning |
| --- | ---: | --- |
| `processing_time/avg` | `10.87 s` | Average processing time for Rollout tasks |
| `processing_time/tp50` | `10.02 s` | 50% of tasks finish within this time |
| `processing_time/tp95` | `19.82 s` | 95% of tasks finish within this time |
| `processing_time/tp99` | `25.70 s` | 99% of tasks finish within this time |
| `processing_time/max` | `96.07 s` | Processing time of the slowest task |

Most Rollout tasks finish in roughly 20 seconds, while the slowest task takes about 96 seconds, indicating the presence of a small number of long-tail samples.

This is exactly the kind of imbalance Fully Async Training is designed to reduce: shorter Rollout tasks can keep producing samples without waiting for the slowest task to finish before Training begins.

#### Sample Staleness

Fully Async Training allows the Trainer to consume a controlled proportion of samples generated by older Policy versions:

| Metric | Example | Meaning |
| --- | ---: | --- |
| `staleness_threshold` | `0.1` | Maximum allowed proportion of stale samples |
| `staleness_samples` | `352.75` | Samples counted as stale in the current statistics window |
| `stale_trajectory_processed` | `80784` | Cumulative number of stale trajectories processed by the Trainer |
| `dropped_stale_samples` | `0` | Number of stale samples dropped because of freshness constraints |

Each sample generates `rollout.n=16` trajectories, so:

```text
stale_trajectory_processed
```

counts trajectories rather than original Prompts. It is also a cumulative metric, so it is expected to increase throughout training. Its absolute value alone is not enough to determine whether staleness is too high.

Here, `dropped_stale_samples=0`, which means no samples have been discarded for exceeding the freshness constraint. To judge whether staleness is affecting training, you should also monitor validation accuracy, reward, and Policy KL trends.

In verl's Fully Async design, a larger `staleness_threshold` makes it easier for Rollout and Training to remain fully overlapped, but it also allows the Trainer to consume more data generated by older Policy versions. In practice, this parameter balances training throughput against Policy Freshness. See the [verl Fully Async Policy documentation](https://github.com/verl-project/verl/blob/main/docs/advance/fully_async.md).

#### Policy Update Stability

| Metric | Example | Meaning |
| --- | ---: | --- |
| `actor/pg_loss` | `0.01085` | Actor Policy Gradient Loss |
| `actor/ppo_kl` | `0.00086` | Approximate KL divergence between the current Policy and the Rollout Policy |
| `actor/pg_clipfrac` | `0.000995` | Fraction of tokens whose PPO ratio triggered clipping |
| `actor/pg_clipfrac_lower` | `0` | Fraction that triggered the lower clipping bound |
| `actor/grad_norm` | `0.233` | Actor gradient norm |
| `actor/lr` | `1e-6` | Current learning rate |

The current `ppo_kl` and `pg_clipfrac` values are both small, suggesting that this Policy Update is relatively conservative and that most probability ratios did not trigger clipping.

`grad_norm=0.233` is below the configured:

```text
grad_clip = 1.0
```

so there is no obvious sign of gradient explosion at this point.

These metrics are most useful when interpreted as trends:

- A sudden large increase in `ppo_kl` or `pg_clipfrac` may indicate that Policy Updates are becoming too large;
- A persistently abnormal increase in `grad_norm` may indicate training instability;
- If these metrics remain close to 0 for a long time while reward and validation accuracy also fail to improve, Policy Updates may be too conservative.

The sign or absolute magnitude of `pg_loss` alone does not directly indicate model quality. It should be interpreted together with reward, KL, gradient norm, and validation metrics.

#### Reward and Advantage

| Metric | Example | Meaning |
| --- | ---: | --- |
| `critic/score/mean` | `-0.079` | Average raw score of the current Training Batch |
| `critic/rewards/mean` | `-0.079` | Average reward after processing by the Reward Manager |
| `critic/advantages/mean` | `-0.027` | Mean Group Relative Advantage |
| `critic/advantages/max` | `3.75` | Maximum advantage in the current batch |
| `critic/advantages/min` | `-3.75` | Minimum advantage in the current batch |

An `advantages/mean` close to 0 is consistent with the expected behavior of Group Relative Advantage because Responses for the same Prompt are normalized relative to their within-group rewards.

A single-step value of `score/mean` or `rewards/mean` does not directly represent final model quality. More importantly, watch whether these values improve over time and evaluate them together with AIME 2024 validation accuracy.

#### Validation Accuracy

`val-core/math_dapo/acc/mean@1` is the primary validation metric for mathematical problem-solving performance:

| Metric | Meaning |
| --- | --- |
| `val-core/math_dapo/acc/mean@1` | Mean accuracy on the `math_dapo` validation set, using 1 Response per Prompt |

The metric name can be read as follows:

- `val-core`: core validation metric;
- `math_dapo`: validation data and Reward Function used;
- `acc`: answer accuracy;
- `mean@1`: generate 1 Response per Prompt and compute the mean accuracy across all validation samples.

For example:

```text
val-core/math_dapo/acc/mean@1 = 0.30
```

means that the model answers about **30%** of the validation problems correctly at that point in training.

#### Prompt and Response Lengths

| Metric | Example | Meaning |
| --- | ---: | --- |
| `prompt_length/mean` | `172` | Average Prompt length in tokens |
| `prompt_length/max` | `952` | Longest Prompt in the current batch |
| `response_length/mean` | `879` | Average Response length in tokens |
| `response_length/max` | `7881` | Longest Response in the current batch |
| `response_length/clip_ratio` | `0` | Fraction of Responses truncated after reaching the maximum length |
| `response/aborted_ratio` | `0` | Fraction of Responses that were aborted |

The longest Response is currently 7881 tokens, which is below the configured:

```text
max_response_length = 8192
```

At the same time, `response_length/clip_ratio=0`, which means no Response in the current batch was truncated because it reached the maximum length.

During training, pay particular attention to:

- whether `response_length/mean` keeps increasing rapidly;
- whether `response_length/clip_ratio` starts to rise;
- whether `response/aborted_ratio` increases unexpectedly.

If many Responses approach 8192 tokens, the model may be developing overly long reasoning paths, and you should examine whether Overlong Reward Shaping is effectively controlling response length.

#### Step Time and Training Throughput

| Metric | Example | Meaning |
| --- | ---: | --- |
| `timing_s/gen` | `54.38 s` | Time spent by the Trainer obtaining generated data |
| `timing_s/adv` | `0.11 s` | Time spent computing Advantage |
| `timing_s/update_actor` | `574.71 s` | Time spent updating the Actor |
| `timing_s/param_sync` | `2.43 s` | Time spent synchronizing the latest Policy parameters to the Rollout Engines |
| `timing_s/step` | `631.65 s` | Total time for the current Training Step |
| `perf/total_num_tokens` | `8,612,699` | Total number of tokens processed in the current step |
| `perf/throughput` | `3408.82` | Training throughput reported by the framework |
